# 01. Camada Bronze: Ingestão
**Objetivo:** carregar os 9 CSVs originais do dataset Olist (Kaggle, licença CC BY-NC-SA 4.0) do Volume `mvp_olist.bronze.raw_files` para tabelas Delta, **sem nenhuma transformação**.

**Decisões:**
- Todas as colunas são lidas como **texto** (sem inferência de tipo), para preservar o dado bruto da maneira exata que veio. A tipagem/limpeza acontece na etapa Silver.
- São adicionados metadados de controle (proveniência): `_ingestao_timestamp` e `_arquivo_origem`.

In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/mvp_olist/bronze/raw_files"

# nome do arquivo -> nome da tabela bronze
arquivos = {
    "olist_customers_dataset.csv":           "customers",
    "olist_geolocation_dataset.csv":         "geolocation",
    "olist_order_items_dataset.csv":         "order_items",
    "olist_order_payments_dataset.csv":      "order_payments",
    "olist_order_reviews_dataset.csv":       "order_reviews",
    "olist_orders_dataset.csv":              "orders",
    "olist_products_dataset.csv":            "products",
    "olist_sellers_dataset.csv":             "sellers",
    "product_category_name_translation.csv": "category_translation",
}

for arquivo, tabela in arquivos.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)   # tudo como string: bronze = dado bruto
          .option("multiLine", True)      # reviews têm comentários com quebra de linha
          .option("escape", '"')
          .option("encoding", "UTF-8")
          .csv(f"{VOLUME}/{arquivo}")
          .withColumn("_ingestao_timestamp", F.current_timestamp())
          .withColumn("_arquivo_origem", F.col("_metadata.file_path")))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"mvp_olist.bronze.{tabela}"))

    print(f"✔ bronze.{tabela}: {df.count():,} linhas")

✔ bronze.customers: 99,441 linhas
✔ bronze.geolocation: 1,000,163 linhas
✔ bronze.order_items: 112,650 linhas
✔ bronze.order_payments: 103,886 linhas
✔ bronze.order_reviews: 99,224 linhas
✔ bronze.orders: 99,441 linhas
✔ bronze.products: 32,951 linhas
✔ bronze.sellers: 3,095 linhas
✔ bronze.category_translation: 71 linhas


In [0]:
%sql
SHOW TABLES IN mvp_olist.bronze;

database,tableName,isTemporary
bronze,category_translation,false
bronze,customers,false
bronze,geolocation,false
bronze,order_items,false
bronze,order_payments,false
bronze,order_reviews,false
bronze,orders,false
bronze,products,false
bronze,sellers,false
